In [1]:
import pandas as pd
import numpy as np
import os

# =========================
# CONFIG
# =========================
pasta = r"C:\Users\User\Downloads\Sorteio_Copa"

arquivos_excel = [f for f in os.listdir(pasta) if f.endswith(".xlsx")]

if len(arquivos_excel) == 0:
    raise Exception("Nenhum arquivo Excel encontrado.")

caminho_arquivo = os.path.join(pasta, arquivos_excel[0])

# =========================
# LEITURA E LIMPEZA
# =========================
df = pd.read_excel(caminho_arquivo)

df.columns = (
    df.columns
    .str.strip()
    .str.upper()
    .str.replace("Í", "I")
)

df = df.loc[:, ~df.columns.str.contains("UNNAMED")]
df = df[["POTE", "TIME", "PAIS"]]

# =========================
# CRIAR POTES
# =========================
potes = {}

for i in range(1, 5):
    pote = df[df["POTE"] == i][["TIME", "PAIS"]].copy()

    pote = pote.rename(columns={
        "TIME": f"Time{i}",
        "PAIS": f"Pais{i}"
    })

    pote["ID"] = 1
    potes[i] = pote.reset_index(drop=True)

# =========================
# GERAR TODAS POSSIBILIDADES
# =========================
p1, p2, p3, p4 = potes[1], potes[2], potes[3], potes[4]

#p1_poss = p1.sample(frac=1).reset_index(drop=True)
#p2_poss = p2.sample(frac=1).reset_index(drop=True)
#p3_poss = p3.sample(frac=1).reset_index(drop=True)
#p4_poss = p4.sample(frac=1).reset_index(drop=True)
#df_poss = (
#    p1_poss.merge(p2_poss, on="ID")
#      .merge(p3_poss, on="ID")
#      .merge(p4_poss, on="ID")
#)

df_poss = (
    p1.merge(p2, on="ID")
      .merge(p3, on="ID")
      .merge(p4, on="ID"))

# ==================================================
# PRIORIZAÇÃO CONCACAF (Lógica idêntica ao SAS)
# ==================================================
# 1. Geramos um número aleatório para cada combinação
df_poss['SORT_PRIORITY'] = np.random.rand(len(df_poss))

# 2. Se o Time1 (Pote 1) for CONCACAF, dividimos o número por 1000 
# Isso garante que eles fiquem no início após o sort
df_poss.loc[df_poss['Pais1'] == 'CONCACAF', 'SORT_PRIORITY'] = df_poss['SORT_PRIORITY'] / 1000

# 3. Ordenamos o espaço amostral (Menores valores primeiro)
df_poss = df_poss.sort_values(by='SORT_PRIORITY').reset_index(drop=True)

# Removemos a coluna auxiliar para não poluir o resultado
df_poss = df_poss.drop(columns=['SORT_PRIORITY'])

#display(df_poss.head(12))
# Veja que os primeiros grupos agora tendem a ser CONCACAF no Pote 1

#display(df_poss)

# =========================
# VALIDAÇÃO (BOOLEANO)
# =========================
def valida(row):
    pais = [row["Pais1"], row["Pais2"], row["Pais3"], row["Pais4"]]

    if pais[0] != "UEFA" and pais[0] == pais[1]: return False
    if pais[0] != "UEFA" and pais[0] == pais[2]: return False
    if pais[0] != "UEFA" and pais[0] == pais[3]: return False           
    if pais[1] != "UEFA" and pais[1] == pais[2]: return False
    if pais[1] != "UEFA" and pais[1] == pais[3]: return False
    if pais[2] != "UEFA" and pais[2] == pais[3]: return False

    if pais[0] == "UEFA" and pais[0] == pais[1] and pais[1] == pais[2]: return False
    if pais[0] == "UEFA" and pais[0] == pais[1] and pais[1] == pais[3]: return False
    if pais[0] == "UEFA" and pais[0] == pais[2] and pais[2] == pais[3]: return False
    if pais[1] == "UEFA" and pais[1] == pais[2] and pais[2] == pais[3]: return False

    if pais[3] == "CONMEBOL/CONCACAF/AFC" :
        if pais[0] == "CONMEBOL" or pais[1] == "CONMEBOL" or pais[2] == "CONMEBOL" : return False
        if pais[0] == "CONCACAF" or pais[1] == "CONCACAF" or pais[2] == "CONCACAF" : return False
        if pais[0] == "AFC" or pais[1] == "AFC" or pais[2] == "AFC" : return False

    if pais[3] == "OFC/CONCACAF/CAF" :
        if pais[0] == "OFC" or pais[1] == "OFC" or pais[2] == "OFC" : return False
        if pais[0] == "CONCACAF" or pais[1] == "CONCACAF" or pais[2] == "CONCACAF" : return False
        if pais[0] == "CAF" or pais[1] == "CAF" or pais[2] == "CAF" : return False

    if pais[0] != "UEFA" and pais[1] != "UEFA" and pais[2] != "UEFA" and pais[3] != "UEFA" : return False

    return True

df_poss = df_poss[df_poss.apply(valida, axis=1)].reset_index(drop=True)
#display(df_poss)

# embaralha (equivalente ao RAND do SAS)
#df_poss = df_poss.sample(frac=1).reset_index(drop=True)
# =========================
# BACKTRACKING CORRETO
# =========================
tentativas_frustradas = 0

lista_poss = []
indices = []
grupos = []

# nível inicial
lista_poss.append(df_poss.copy())
indices.append(0)

nivel = 0

,Time1,Pais1,ID,Time2,Pais2,Time3,Pais3,Time4,Pais4
0,Canadá,CONCACAF,1,Japão,AFC,Tunísia,CAF,Ucrânia/Suécia/Polônia/Albânia,UEFA
1,Estados Unidos,CONCACAF,1,Uruguai,CONMEBOL,Costa do Marfim,CAF,Ucrânia/Suécia/Polônia/Albânia,UEFA
2,Estados Unidos,CONCACAF,1,Colômbia,CONMEBOL,Noruega,UEFA,Jordânia,AFC
3,Canadá,CONCACAF,1,Senegal,CAF,Noruega,UEFA,Jordânia,AFC
4,Estados Unidos,CONCACAF,1,Austria,UEFA,Africa do Sul,CAF,Dinamarca/Macedônia do Norte/Tchéquia/Irlanda,UEFA
...,...,...,...,...,...,...,...,...,...
8174,Inglaterra,UEFA,1,Austrália,AFC,Argélia,CAF,Ucrânia/Suécia/Polônia/Albânia,UEFA
8175,Holanda,UEFA,1,Equador,CONMEBOL,Costa do Marfim,CAF,Dinamarca/Macedônia do Norte/Tchéquia/Irlanda,UEFA
8176,Holanda,UEFA,1,Irã,AFC,Panamá,CONCACAF,Dinamarca/Macedônia do Norte/Tchéquia/Irlanda,UEFA
8177,Inglaterra,UEFA,1,Coréia do Sul,AFC,Panamá,CONCACAF,Gana,CAF


In [2]:
while nivel < 12:
    df_atual = lista_poss[nivel]

    if indices[nivel] < len(df_atual):
        escolha = df_atual.iloc[indices[nivel]]
        
        # --- CÁLCULO DUPLO UEFA (Lógica SAS) ---
        # Conta UEFAs na escolha atual
        confs_escolha = [escolha["Pais1"], escolha["Pais2"], escolha["Pais3"], escolha["Pais4"]]
        uefas_na_escolha = confs_escolha.count("UEFA")
        
        # Soma com o que já existe nos grupos confirmados
        uefas_nos_anteriores = [sum([g[f"Pais{k}"] == "UEFA" for k in range(1, 5)]) for g in grupos]
        total_duplo_uefa = sum([1 for q in uefas_nos_anteriores if q > 1])
        
        if uefas_na_escolha > 1:
            total_duplo_uefa += 1

        # --- VALIDAÇÃO DA TRAVA (Máx 4 grupos com 2 UEFAs) ---
        if total_duplo_uefa <= 4:
            print(f"✅ Grupo {nivel+1}: {escolha['Time1']} escolhido (Opção {indices[nivel]+1}/{len(df_atual)})")
            
            grupos.append(escolha)

            # RECONSTRÓI A BASE FILTRADA
            times_usados = [t for g in grupos for t in [g["Time1"], g["Time2"], g["Time3"], g["Time4"]]]
            
            # Filtro otimizado: remove combinações que contenham qualquer time já usado
            mask = ~df_poss[['Time1', 'Time2', 'Time3', 'Time4']].isin(times_usados).any(axis=1)
            df_prox = df_poss[mask].reset_index(drop=True)

            lista_poss.append(df_prox)
            indices.append(0)
            nivel += 1
        else:
            # Se furar a trava de 4 grupos Duplo UEFA, incrementa o índice para tentar a próxima opção
            indices[nivel] += 1
            continue

    else:
        # BACKTRACK (Sem opções ou trava de UEFA estourada no nível anterior)
        tentativas_frustradas += 1
        print(f"🔄 Tentativa Frustrada #{tentativas_frustradas} no Grupo {nivel+1}")

        indices[nivel] = 0
        lista_poss.pop()
        indices.pop()
        if grupos: 
            grupos.pop()

        nivel -= 1
        if nivel < 0: raise Exception("Sem solução")
        
        indices[nivel] += 1

print(f"")
print(f"\n✅ GRUPOS GERADOS COM SUCESSO! Total de tentativas frustradas: {tentativas_frustradas}")
# =========================
# RESULTADO FINAL
# =========================
resultado = []

for i, g in enumerate(grupos, 1):
    resultado.append({
        "GRUPO": i,
        "TIME1": f"{g['Time1']} ({g['Pais1']})", "TIME2": f"{g['Time2']} ({g['Pais2']})",
        "TIME3": f"{g['Time3']} ({g['Pais3']})", "TIME4": f"{g['Time4']} ({g['Pais4']})"
    })

resultado = pd.DataFrame(resultado)

# =========================
# RESULTADO FINAL
# =========================
resultado = []

for i, g in enumerate(grupos, 1):
    resultado.append({
        "GRUPO": i,
        "TIME1": f"{g['Time1']} ({g['Pais1']})", "TIME2": f"{g['Time2']} ({g['Pais2']})",
        "TIME3": f"{g['Time3']} ({g['Pais3']})", "TIME4": f"{g['Time4']} ({g['Pais4']})"
    })

resultado = pd.DataFrame(resultado)

print("\nRESULTADO FINAL:\n")
display(resultado)

print(f"\nTOTAL DE TENTATIVAS FRUSTRADAS: {tentativas_frustradas}")

✅ Grupo 1: Canadá escolhido (Opção 1/8179)
✅ Grupo 2: Estados Unidos escolhido (Opção 1/5792)
✅ Grupo 3: México escolhido (Opção 1/4050)
✅ Grupo 4: Portugal escolhido (Opção 1/2789)
✅ Grupo 5: Bélgica escolhido (Opção 1/1662)
✅ Grupo 6: Holanda escolhido (Opção 1/932)
✅ Grupo 7: Inglaterra escolhido (Opção 1/488)
✅ Grupo 8: França escolhido (Opção 1/265)
✅ Grupo 9: Alemanha escolhido (Opção 1/102)
✅ Grupo 10: Brasil escolhido (Opção 2/30)
✅ Grupo 11: Argentina escolhido (Opção 2/7)
❌ Tentativa Frustrada #1 no Grupo 12
✅ Grupo 11: Argentina escolhido (Opção 3/7)
❌ Tentativa Frustrada #2 no Grupo 12
✅ Grupo 11: Espanha escolhido (Opção 5/7)
❌ Tentativa Frustrada #3 no Grupo 12
✅ Grupo 11: Argentina escolhido (Opção 7/7)
❌ Tentativa Frustrada #4 no Grupo 12
❌ Tentativa Frustrada #5 no Grupo 11
✅ Grupo 10: Argentina escolhido (Opção 4/30)
✅ Grupo 11: Brasil escolhido (Opção 1/4)
❌ Tentativa Frustrada #6 no Grupo 12
✅ Grupo 11: Brasil escolhido (Opção 4/4)
❌ Tentativa Frustrada #7 no Grupo 


RESULTADO FINAL:



,GRUPO,TIME1,TIME2,TIME3,TIME4
0,1,Canadá (CONCACAF),Japão (AFC),Tunísia (CAF),Ucrânia/Suécia/Polônia/Albânia (UEFA)
1,2,Estados Unidos (CONCACAF),Colômbia (CONMEBOL),Noruega (UEFA),Jordânia (AFC)
2,3,México (CONCACAF),Coréia do Sul (AFC),Africa do Sul (CAF),Turquia/Romênia/Eslováquia/Kosovo (UEFA)
3,4,Portugal (UEFA),Uruguai (CONMEBOL),Catar (AFC),Dinamarca/Macedônia do Norte/Tchéquia/Irlanda ...
4,5,Bélgica (UEFA),Equador (CONMEBOL),Costa do Marfim (CAF),Haiti (CONCACAF)
5,6,Holanda (UEFA),Austria (UEFA),Panamá (CONCACAF),Cabo Verde (CAF)
6,7,Inglaterra (UEFA),Marrocos (CAF),Escócia (UEFA),Bolívia/Suriname/Iraque (CONMEBOL/CONCACAF/AFC)
7,8,França (UEFA),Suiça (UEFA),Arábia Saudita (AFC),Gana (CAF)
8,9,Brasil (CONMEBOL),Austrália (AFC),Egito (CAF),Itália/Irlanda do Norte/País de Gales/Bósnia (...
9,10,Espanha (UEFA),Irã (AFC),Argélia (CAF),Curaçao (CONCACAF)



TOTAL DE TENTATIVAS FRUSTRADAS: 79


In [27]:
# --- CAMADA DE AUDITORIA E VALIDAÇÃO DETALHADA ---

# 1. Preparação dos dados de auditoria
df_audit = pd.DataFrame(resultado)

for i in range(1, 5):
    col = f'TIME{i}'
    # O .str.contains verifica a presença da sigla (insensível a maiúsculas/minúsculas para evitar erros)
    df_audit[f'U_{i}'] = df_audit[col].str.contains('UEFA', case=False, na=False)
    df_audit[f'CM_{i}'] = df_audit[col].str.contains('CONMEBOL', case=False, na=False)
    df_audit[f'CC_{i}'] = df_audit[col].str.contains('CONCACAF', case=False, na=False)
    df_audit[f'A_{i}'] = df_audit[col].str.contains('AFC', case=False, na=False)
    df_audit[f'CF_{i}'] = df_audit[col].str.contains(r'\bCAF\b', case=False, na=False)
#    df_audit[f'CF_2{i}'] = df_audit[col].str.contains('/CAF', case=False, na=False)
    df_audit[f'O_{i}'] = df_audit[col].str.contains('OFC', case=False, na=False)

# 2. Contagem por Confederação (Soma das colunas booleanas: True=1, False=0)
df_audit['QTD_UEFA']     = df_audit[[f'U_{i}' for i in range(1, 5)]].sum(axis=1)
df_audit['QTD_CONMEBOL'] = df_audit[[f'CM_{i}' for i in range(1, 5)]].sum(axis=1)
df_audit['QTD_CONCACAF'] = df_audit[[f'CC_{i}' for i in range(1, 5)]].sum(axis=1)
df_audit['QTD_AFC']      = df_audit[[f'A_{i}' for i in range(1, 5)]].sum(axis=1)
df_audit['QTD_CAF']      = df_audit[[f'CF_{i}' for i in range(1, 5)]].sum(axis=1)
df_audit['QTD_OFC']      = df_audit[[f'O_{i}' for i in range(1, 5)]].sum(axis=1)

# 3. Validação das Regras (Simulando o CASE WHEN com np.where)
# Regra UEFA: Mínimo 1 e Máximo 2
df_audit['VRF_UEFA'] = np.where((df_audit['QTD_UEFA'] >= 1) & (df_audit['QTD_UEFA'] <= 2), '✅', '❌')

# Regras Demais: Máximo 1 por grupo
df_audit['VRF_CONMEBOL'] = np.where(df_audit['QTD_CONMEBOL'] <= 1, '✅', '❌')
df_audit['VRF_CONCACAF'] = np.where(df_audit['QTD_CONCACAF'] <= 1, '✅', '❌')
df_audit['VRF_AFC']      = np.where(df_audit['QTD_AFC'] <= 1, '✅', '❌')
df_audit['VRF_CAF']      = np.where(df_audit['QTD_CAF'] <= 1, '✅', '❌')
df_audit['VRF_OFC']      = np.where(df_audit['QTD_OFC'] <= 1, '✅', '❌')

# 4. Exibição da Tabela de Auditoria (A foto 4 do seu post)
colunas_visualizacao = ['GRUPO', 'VRF_UEFA', 'VRF_CONMEBOL', 'VRF_CONCACAF', 'VRF_AFC', 'VRF_CAF']
print("\n--- RELATÓRIO DE AUDITORIA POR GRUPO ---")
print(df_audit[colunas_visualizacao].to_string(index=False))

# 5. Check de Unicidade Global (Corrigido para DataFrame)
# Pegamos todas as colunas de times e transformamos em uma lista única (flat)
todas_selecoes = resultado[["TIME1", "TIME2", "TIME3", "TIME4"]].values.flatten().tolist()

# Verificamos se o total de itens é igual ao total de itens únicos
times_duplicados = [t for t in set(todas_selecoes) if todas_selecoes.count(t) > 1]
status_unicidade = "OK ✅" if not times_duplicados else f"ERRO (Duplicados: {list(set(times_duplicados))}) ❌"

print(f"Check de Unicidade Global: {status_unicidade}")

# 6. Resumo Final
if 'ERRO ❌' in df_audit[colunas_visualizacao].values or "ERRO" in status_unicidade:
    print("\n🚨 ATENÇÃO: O sorteio contém violações de regras!")
else:
    print("\n🏆 SUCESSO: Todas as restrições foram respeitadas em todos os grupos.")


--- RELATÓRIO DE AUDITORIA POR GRUPO ---
 GRUPO VRF_UEFA VRF_CONMEBOL VRF_CONCACAF VRF_AFC VRF_CAF
     1        ✅            ✅            ✅       ✅       ✅
     2        ✅            ✅            ✅       ✅       ✅
     3        ✅            ✅            ✅       ✅       ✅
     4        ✅            ✅            ✅       ✅       ✅
     5        ✅            ✅            ✅       ✅       ✅
     6        ✅            ✅            ✅       ✅       ✅
     7        ✅            ✅            ✅       ✅       ✅
     8        ✅            ✅            ✅       ✅       ✅
     9        ✅            ✅            ✅       ✅       ✅
    10        ✅            ✅            ✅       ✅       ✅
    11        ✅            ✅            ✅       ✅       ✅
    12        ✅            ✅            ✅       ✅       ✅
Check de Unicidade Global: OK ✅

🏆 SUCESSO: Todas as restrições foram respeitadas em todos os grupos.
